<a href="https://colab.research.google.com/github/Vermont-Complex-Systems/storywrangler/blob/main/notebooks/country_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Country timeseries analysis*

In this notebook, we show how the storywrangler SDK ease the analysis of large scale text corpora.

In [ ]:
!pip install -U storywrangler

In [ ]:
# local packages
from storywrangler import Storywrangler

import matplotlib.cm as cm
import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
import storywrangler
storywrangler.__version__

'0.0.21'

In [ ]:
client=Storywrangler()

In [ ]:
wiki = client.dataset("wikimedia", "ngrams")

In [ ]:
wiki.summary.df()

,dimension,kind,default,values,min,max
0,country,entity,None,entity IDs — resolve via .adapter,None,None
1,date,time,None,dates: range,2024-09-01,2026-07-25
2,ngram_size,filter,1,"1, 2",None,None
3,granularity,filter,daily,"daily, monthly, weekly",None,None


In [ ]:
timerange = "2024-09-30,2026-07-25"

# countries with big enough wiki english
countries = {
    "us": "United States", "ca": "Canada", "au": "Australia",
    "uk": "United Kingdom", "india": "India", "fi": "Finland",
    "no": "Norway", "ph": "Philippines", "ro": "Romania", 'bu': "Bulgaria"
}


Trumpology

In [ ]:
series = {}
for code, entity in countries.items():
    s = (wiki.term_series("Donald Trump", entity=entity, ngram_size=2, dates=timerange)
             .df().set_index("date")["rank"])
    s.index = pd.to_datetime(s.index)
    series[f"rank_{code}"] = s

multi_df_us = pd.concat(series, axis=1).sort_index()

In [ ]:
multi_df_us.head()

,rank_us,rank_ca,rank_au,rank_uk,rank_india,rank_fi,rank_no,rank_ph,rank_ro,rank_bu
date,,,,,,,,,,
2024-09-30,1404,1607.0,1947,2778,5422,3362.0,2432,3872.0,3362,3597.0
2024-10-01,1185,1341.0,1640,2373,5269,1409.0,712,3517.0,2295,1204.0
2024-10-02,818,977.0,1392,2197,4225,940.0,978,2705.0,1558,781.0
2024-10-03,995,1042.0,1511,2450,4705,2673.0,843,2526.0,2336,3633.0
2024-10-04,1028,1066.0,1516,2298,4372,1443.0,1300,2461.0,2446,1658.0


In [ ]:
series = {}
for code, entity in countries.items():
    s = (wiki.term_series("Mark Carney", entity=entity, ngram_size=2, dates=timerange)
             .df().set_index("date")["rank"])
    s.index = pd.to_datetime(s.index)
    series[f"rank_{code}"] = s

multi_df_can = pd.concat(series, axis=1).sort_index()

## Plotting

### Helpers

In [97]:
def _weekly_band(df):
    """Wide rank_*-style df -> long weekly geometric mean + ×/÷ gstd band."""
    grp = np.log(df).resample("W")
    mean_, std_ = grp.mean(), grp.std()
    def melt(frame, name):
        return frame.melt(ignore_index=False, var_name="country", value_name=name).reset_index()
    weekly = melt(np.exp(mean_), "center")
    weekly["lower"] = melt(np.exp(mean_ - std_), "lower")["lower"]
    weekly["upper"] = melt(np.exp(mean_ + std_), "upper")["upper"]
    weekly["country"] = weekly["country"].str.removeprefix("rank_").str.upper()
    return weekly

def _band_chart(weekly, *, y_title, title, reverse=False, baseline=None,
                width=780, height=380):
    ylog  = alt.Scale(type="log", reverse=reverse)
    color = alt.Color("country:N", title="country", scale=alt.Scale(scheme="tableau20"))
    legend_sel = alt.selection_point(fields=["country"], bind="legend")            # click legend
    line_sel   = alt.selection_point(fields=["country"], on="click",
                                     nearest=True, clear="dblclick")               # click a line
    active = legend_sel & line_sel        # empty=True default -> all bright until you pick one
    tip = ["country:N", alt.Tooltip("date:T"),
           alt.Tooltip("center:Q", format=".2f", title="value")]
    base = alt.Chart(weekly)
    band = base.mark_area().encode(
        x=alt.X("date:T", title="week"),
        y=alt.Y("lower:Q", scale=ylog, title=y_title),
        y2="upper:Q", color=color,
        opacity=alt.condition(active, alt.value(0.15), alt.value(0.0)), tooltip=tip)
    line = base.mark_line(point=alt.OverlayMarkDef(size=16)).encode(
        x="date:T", y=alt.Y("center:Q", scale=ylog), color=color,
        opacity=alt.condition(active, alt.value(1.0), alt.value(0.12)), tooltip=tip)
    layers = [band, line]
    if baseline is not None:
        rule = alt.Chart(pd.DataFrame({"y": [baseline]})).mark_rule(
            strokeDash=[4, 4], color="gray").encode(y="y:Q")
        layers = [band, rule, line]
    return (alt.layer(*layers).add_params(legend_sel, line_sel)
            .properties(width=width, height=height, title=title).interactive())

# ============================ public API ============================
def plot_rank(df, title="Weekly search rank with variability band, by country", **kw):
    """Plot raw ranks: reversed log axis (rank 1 on top), variability band."""
    return _band_chart(_weekly_band(df), reverse=True, baseline=None,
                       y_title="rank (geometric mean ×/÷ gstd, log)",
                       title=title, width=760, height=360, **kw)

def plot_relative_interest(df, term="Trump", **kw):
    """Ranks -> relative interest (1/rank, indexed to each country's median), then plot."""
    interest = 1 / df
    rel = interest / interest.median()
    return _band_chart(_weekly_band(rel), reverse=False, baseline=1,
                       y_title="relative interest  (1 = country's typical level)",
                       title=f'Relative interest in "{term}" by country '
                             f'(weekly, log; 1 = country baseline)', **kw)

Tip: You can click to filter

## Trumpology

In [ ]:
# How multiple countries are bumping into the "Donald Trump" Name across wiki pages
plot_rank(multi_df_us).display()

alt.LayerChart(...)

> Tip to use the plot; you can click to highlight particular countries, click elsewhere to come back, scroll to zoom, and double click elsewhere to recenter.

A rank is really a stand-in for share of attention, and under a Zipf/power-law view of pageviews, that share is roughly proportional to 1/rank. So the natural transform is:
- 1 / rank → an attention proxy where higher = more interest, bounded in (0, 1], and top positions dominate the way they should.
- Divide each country by its own typical level (median), so the unit becomes "how does this compare to normal for this country" — which also removes the baseline differences between big and small countries so their shapes are comparable.

Note that the height of a relative spike in the plots that will follow is inversely related to how much a country normally cares. A country where the figure is usually background noise has a low baseline, so a breakthrough week launches it to 20×; a country that always pays attention has a high baseline and shows smaller relative swings even during the same event. So the tallest spikes are probably not the countries most engaged — they're the ones for whom the figure is normally off the radar.

In [ ]:
plot_relative_interest(multi_df_us, term="Donald Trump").display()

alt.LayerChart(...)

Overall we can see some synchronization in spikes, with what seems to be large spikes. It is not secret, the world is holding it's breath whenever Trump does something crazy. We see
- November 2024 spike is the US election
- the early-2025 one lines up with the inauguration
- February 28, 2026, when the US joined Israel in launching major strikes on Iran

What about some other prime ministers?

## Matrix of national interests

In [ ]:
# short label -> full ngram term
leaders = {"Carney": "Mark Carney", "Starmer": "Keir Starmer", "Trump": "Donald Trump"}

def fetch_leader(term, countries, timerange, ngram_size=None):
    """One wide rank_* frame (countries as columns) for a single leader/term."""
    ngram_size = ngram_size or len(term.split())      # 'Donald Trump' -> 2, computed automatically
    series = {}
    for code, entity in countries.items():
        s = (wiki.term_series(term, entity=entity, ngram_size=ngram_size, dates=timerange)
                 .df().set_index("date")["rank"])
        s.index = pd.to_datetime(s.index)
        series[f"rank_{code}"] = s
    return pd.concat(series, axis=1).sort_index()

leader_dfs = {name: fetch_leader(term, countries, timerange) for name, term in leaders.items()}

In [ ]:
# here the question is:
# Timing and synchronization and each country's own responsiveness.
# That is, "when did attention move, did it move together,
# and how big was each country's move relative to its own normal"
for name, df in leader_dfs.items():
    plot_relative_interest(df, term=leaders[name]).display()

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

The Carney chart is noisier and much less coupled. Instead of one worldwide bundle, you get a sustained elevation led by one country
- the Canadian series riding above 1 through roughly the spring-2025 stretch that maps onto his Liberal leadership win (March 2025) and the federal election (April/May 2025) (duh)

The next question we get rid of the median normalization. Here we answer "who cares most".

In [100]:
df = leader_dfs["Carney"]
comparable = df.loc[:, df.notna().mean() >= 0.6]      # keep countries present ≥60% of days
plot_rank(comparable, title='Search rank for "Mark Carney" (countries with ≥60% coverage)')

alt.LayerChart(...)

Canada sits about an order of magnitude above everyone else, persistently. CA rides around rank ~10,000–20,000 the whole window, while US, UK, India, PH, and AU cluster together down around rank ~100,000–200,000. That gap is the magnitude signal the relative-interest chart structurally erased — there, every country orbited 1 and Canada looked like just another line. Here it's unambiguous: for its own PM, Canada is the dramatically most-engaged country, which is the direct answer to the "does each country care most about its own leader" question. Emphatically yes.

In [105]:
# Clearly Keir Starmer was not as visible as Carney was within its home country
# and now he is gone.
df = leader_dfs["Starmer"]
comparable = df.loc[:, df.notna().mean() >= 0.6]      # keep countries present ≥60% of days
plot_rank(comparable, title='Search rank for "Keir Starmer" (countries with ≥60% coverage)')

alt.LayerChart(...)

In [104]:
# back to Trump, for reference. How Trump makes it very correlated
df = leader_dfs["Trump"]
comparable = df.loc[:, df.notna().mean() >= 0.6]      # keep countries present ≥60% of days
plot_rank(comparable, title='Search rank for "Donald Trump" (countries with ≥60% coverage)')

alt.LayerChart(...)